# 8 · Orthographic recognition with **W2v-BERT 2.0**

The second of the two training targets. Stage 5 fine-tunes **XLS-R-300M** to
emit **IPA phonemes**; this one fine-tunes **`facebook/w2v-bert-2.0`** to emit
**Kölsch orthography** — the spelling used in *Alles Kölsch*.

| | stage 5 | stage 8 |
|---|---|---|
| encoder | XLS-R-300M (~315M) | w2v-BERT 2.0 (~580M) |
| input | raw waveform | 80-bin log-mel, stride 2 |
| target | IPA phoneme | grapheme |
| headline metric | PER | **CER** |

## Read this before you compare the two

**There is no German warm-start here.** Stage 5 starts from a checkpoint already
adapted to German; `facebook/w2v-bert-2.0` is a raw pretrained model with no CTC
head, so both `lm_head` **and** the conv adapter initialise randomly. The
stronger multilingual pretraining has to pay for that. Do not assume the bigger
model wins — measure it.

**CER is the headline, not WER.** Kölsch has no standardised orthography, so
*janz*/*ganz* and *zusamme*/*zosamme* are the same word spelled two ways. WER
charges full price for that; CER degrades gracefully. A scoring-only variant
folding is reported *alongside* the raw numbers below — never instead of them.


In [ ]:
!pip -q install torch torchaudio transformers datasets evaluate jiwer librosa soundfile pandas matplotlib
import torch, numpy as np, pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "·", device,
      "·", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")


In [ ]:
# === Portable setup — identical paths in VS Code, Jupyter & Google Colab ===
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    _here = any((Path(p)/"kolsch_paths.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
    if not _here and not Path("/content/kolsch-tandem/kolsch_paths.py").exists():
        os.system("git clone -q https://github.com/chemvatho/Koelsch-Phoneme-Recognition.git /content/kolsch-tandem")
except Exception:
    pass

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"kolsch_paths.py").exists()),
             Path("/content/kolsch-tandem"))
sys.path.insert(0, str(_root))
from kolsch_paths import ROOT, DATA, AUDIO, TRANS, SEG, INDEX, LEXICON, MODELS
os.chdir(ROOT)
print("repo root:", ROOT)


## Config — one block, every knob

In [ ]:
import re, json, hashlib
from dataclasses import dataclass
from transformers import set_seed

os.environ["WANDB_DISABLED"] = "true"

@dataclass
class CFG:
    seed: int = 42
    model_id: str = "facebook/w2v-bert-2.0"
    sampling_rate: int = 16_000

    # add_adapter=True is the standard CTC recipe for w2v-BERT 2.0. It also HALVES
    # the frame rate, 20ms -> 40ms, which is irrelevant for WER/CER and very
    # relevant if you feed this model to notebook 9 -- see the closing note.
    add_adapter: bool = True

    metric_for_best: str = "cer"          # "cer" | "wer"
    report_canonical: bool = True         # fold spelling variants FOR SCORING ONLY

    # SMOKE=True runs two optimiser steps on a handful of clips: it proves the
    # pipeline is wired up without pretending to be a training run.
    smoke: bool = bool(int(os.environ.get("KOLSCH_SMOKE", "0")))
    epochs: int = 30
    batch_size: int = 4
    grad_accum: int = 4
    lr: float = 5e-5

cfg = CFG()
set_seed(cfg.seed)
OUT_DIR = os.path.join(MODELS, "kolsch_w2vbert_orthography")
print("model:", cfg.model_id, "| smoke:", cfg.smoke, "| out:", OUT_DIR)


## Data — the repo manifest

Same source and the same speaker-disjoint split as stages 5 and 7, so the three
targets are comparable. The split hashes the **speaker**, not the row: a speaker
in train must never appear in test, or the score measures memorisation.

With only one speaker in the shipped example there is nothing to hold out, so
the code falls back to a random split and says so. That is fine for a smoke run
and worthless as an evaluation — scale up `data/` before believing any number.


In [ ]:
MANIFEST = os.path.join(SEG, "manifest.csv")
assert os.path.exists(MANIFEST), f"{MANIFEST} not found — run notebook 3 first."
man = pd.read_csv(MANIFEST)
man = man.merge(pd.read_csv(INDEX)[["id", "speaker"]], on="id", how="left")

def bucket(sp):
    return {0: "test", 1: "valid"}.get(
        int(hashlib.md5(str(sp).encode()).hexdigest(), 16) % 10, "train")

speakers = man["speaker"].fillna("unknown").unique()
SPEAKER_DISJOINT = len(speakers) >= 3
if SPEAKER_DISJOINT:
    man["split"] = man["speaker"].map(bucket)
else:
    print(f"!! only {len(speakers)} speaker(s) — falling back to a RANDOM split.")
    print("!! test and train share a voice, so the score is not a generalisation estimate.")
    r = np.random.default_rng(cfg.seed).random(len(man))
    man["split"] = np.where(r < 0.8, "train", np.where(r < 0.9, "valid", "test"))
for s in ("valid", "test"):                     # never leave a split empty
    if (man["split"] == s).sum() == 0:
        man.loc[man.index[-1 if s == "valid" else -2], "split"] = s
print({s: int((man["split"] == s).sum()) for s in ("train", "valid", "test")})


## Cleaning — the glue policy

Apostrophes are **dropped, not spaced**: `d'r` → `dr`, `si'mer` → `simer`. Kölsch
elision writes one spoken word with an apostrophe inside it, so splitting there
would invent a word boundary the speaker never produced.


In [ ]:
_APOSTROPHES = "'\u2019\u02bc\u02bb`"

def clean_orthography(text):
    """Kölsch orthographic normalisation for grapheme CTC (glue policy)."""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    for a in _APOSTROPHES:
        text = text.replace(a, "")                  # GLUE: pieces join up
    text = re.sub(r"[^a-zäöüß ]+", " ", text)       # keep Kölsch letters + space
    return re.sub(r"\s+", " ", text).strip()

for s in ["so jeht dat nit!", "wo si'mer dann he?", "d'm wor't ni'mih"]:
    print(f"{s!r:32} -> {clean_orthography(s)!r}")

man["label"] = man["text"].map(clean_orthography)
man = man[man["label"].str.len() > 0].reset_index(drop=True)
print(len(man), "usable rows")


## Variant folding — **for scoring only**

Folds *janz*/*ganz* and friends to one key before WER/CER, so the metric stops
charging for a spelling choice. **Never applied to training labels**, and the
raw numbers stay the primary figure. The guard asserts that real minimal pairs
are not collapsed — the fold is deliberately conservative, and `ja`/`jo` are
kept apart by design.


In [ ]:
_CANON_EXPLICIT = {
    "zusamme": "zosamme", "zusammen": "zosammen",
    "avver": "ävver", "äver": "ävver",
    "dan": "dann", "wider": "widder", "nitt": "nit",
    "kunt": "kunnt", "alsu": "also", "vrau": "frau",
}

def _canon_word(w):
    if not w:
        return w
    if w in _CANON_EXPLICIT:
        return _CANON_EXPLICIT[w]
    if w.startswith("zusamme"):
        return "zosamme" + w[7:]
    # Kölsch <g> -> [j]: participles (ge- -> je-), pre-vocalic g-, gr-/gl- clusters.
    # Folding the clusters is safe because we only need ONE SHARED KEY -- this is
    # not a claim about which spelling is correct.
    if w.startswith("ge") and len(w) > 3:
        return "je" + w[2:]
    if len(w) > 1 and w[0] == "g" and w[1] in "aeiouäöü":
        return "j" + w[1:]
    if w.startswith(("gr", "gl")):
        return "j" + w[1:]
    return w

def canonicalize(text):
    return " ".join(_canon_word(t) for t in str(text).split() if t)

for _a, _b in [("jott", "jot"), ("denn", "den"), ("herr", "her"), ("sonn", "son"),
               ("kamm", "kam"), ("schon", "schön"), ("ja", "jo")]:
    assert _canon_word(_a) != _canon_word(_b), f"wrongly merged {_a}/{_b}"
print("minimal-pair guard OK  ·", canonicalize("dat wor janz jot"))


## Vocabulary — from **train only**

In [ ]:
chars = set()
for s in man.loc[man["split"] == "train", "label"]:
    chars.update(s.replace(" ", ""))
vocab = {c: i for i, c in enumerate(sorted(chars))}
vocab["|"] = len(vocab); vocab["[UNK]"] = len(vocab); vocab["[PAD]"] = len(vocab)

VOCAB_PATH = os.path.join(MODELS, "vocab_ortho_w2vbert.json")
json.dump(vocab, open(VOCAB_PATH, "w", encoding="utf-8"), ensure_ascii=False)
print(f"{len(vocab)} tokens ->", VOCAB_PATH)

unseen = set().union(*[set(s.replace(" ", "")) for s in
                       man.loc[man["split"] != "train", "label"]]) - chars
print("characters in valid/test but not in train:", unseen or "none")


## The w2v-BERT feature stack

This is the one place the port really differs from stage 5. w2v-BERT eats
**log-mel filterbank features**, not raw waveform, so
`Wav2Vec2FeatureExtractor` → `SeamlessM4TFeatureExtractor` and
`Wav2Vec2Processor` → `Wav2Vec2BertProcessor`. Downstream that renames
`input_values` → `input_features` everywhere.


In [ ]:
from transformers import (Wav2Vec2CTCTokenizer, SeamlessM4TFeatureExtractor,
                          Wav2Vec2BertProcessor)

tokenizer = Wav2Vec2CTCTokenizer(VOCAB_PATH, unk_token="[UNK]", pad_token="[PAD]",
                                 word_delimiter_token="|")
feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(cfg.model_id)
processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

print("input names:", processor.feature_extractor.model_input_names,
      "| mels:", feature_extractor.num_mel_bins,
      "| stride:", feature_extractor.stride)

# Preflight: labels must tokenize without [UNK] and round-trip exactly.
# group_tokens=False is REQUIRED — the default CTC-collapses doubled letters
# ('hatt' -> 'hat'), which is right for predictions and wrong for checking labels.
_unk, _tot, _n = processor.tokenizer.unk_token_id, 0, 0
for _t in man["label"]:
    _ids = processor.tokenizer(_t).input_ids
    _tot += len(_ids); _n += sum(1 for i in _ids if i == _unk)
_probe = man["label"].iloc[0]
_rt = processor.tokenizer.decode(processor.tokenizer(_probe).input_ids, group_tokens=False)
print(f"[UNK] rate: {_n}/{_tot} = {_n/max(_tot,1):.4%} | round-trip:",
      "OK" if _rt == _probe else f"MISMATCH {_probe!r} -> {_rt!r}")
assert _rt == _probe, "label round-trip failed"


## Dataset and collator

In [ ]:
import soundfile as sf, librosa
from datasets import Dataset

def _load_16k(path):
    wav, sr = sf.read(path)
    if getattr(wav, "ndim", 1) > 1:
        wav = wav.mean(axis=1)
    wav = np.asarray(wav, dtype=np.float32)
    return librosa.resample(wav, orig_sr=sr, target_sr=cfg.sampling_rate) \
        if sr != cfg.sampling_rate else wav

def prepare(batch):
    feats = processor(_load_16k(batch["audio_path"]),
                      sampling_rate=cfg.sampling_rate).input_features[0]
    batch["input_features"] = feats
    batch["labels"] = processor.tokenizer(batch["label"]).input_ids
    return batch

def to_ds(split):
    sub = man[man["split"] == split][["audio_path", "label"]]
    return Dataset.from_pandas(sub, preserve_index=False).map(
        prepare, remove_columns=["audio_path", "label"])

ds = {s: to_ds(s) for s in ("train", "valid", "test")}
print({s: len(d) for s, d in ds.items()},
      "| feature dim:", np.asarray(ds["train"][0]["input_features"]).shape)


In [ ]:
from dataclasses import field
from typing import Any

class Collator:
    """Pads input_features and labels separately; label pad -> -100 so CTC ignores it."""
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        batch["labels"] = labels["input_ids"].masked_fill(
            labels.attention_mask.ne(1), -100)
        return batch

collator = Collator(processor)
print("collator ready ·", list(collator([ds["train"][0], ds["train"][1]]).keys()))


## Metrics — raw first, canonical alongside

In [ ]:
import evaluate
wer_metric, cer_metric = evaluate.load("wer"), evaluate.load("cer")

def compute_metrics(pred):
    ids = np.argmax(pred.predictions, axis=-1)
    labels = np.where(pred.label_ids != -100, pred.label_ids, processor.tokenizer.pad_token_id)
    hyp = processor.batch_decode(ids)
    ref = processor.batch_decode(labels, group_tokens=False)
    keep = [i for i, r in enumerate(ref) if r.strip()]
    hyp, ref = [hyp[i] for i in keep], [ref[i] for i in keep]
    if not ref:
        return {"wer": float("nan"), "cer": float("nan")}
    out = {"wer": wer_metric.compute(predictions=hyp, references=ref),
           "cer": cer_metric.compute(predictions=hyp, references=ref)}
    if cfg.report_canonical:
        ch, cr = [canonicalize(h) for h in hyp], [canonicalize(r) for r in ref]
        out["wer_canonical"] = wer_metric.compute(predictions=ch, references=cr)
        out["cer_canonical"] = cer_metric.compute(predictions=ch, references=cr)
    return out
print("metrics ready — raw wer/cer are the primary figures")


## Model

In [ ]:
from transformers import Wav2Vec2BertForCTC

model = Wav2Vec2BertForCTC.from_pretrained(
    cfg.model_id,
    attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0,
    mask_time_prob=0.0, layerdrop=0.0,
    ctc_loss_reduction="mean",
    add_adapter=cfg.add_adapter,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
).to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M params "
      f"| adapter={cfg.add_adapter} | lm_head and adapter are randomly initialised")


## Training

`KOLSCH_SMOKE=1` runs two steps to prove the wiring. A real run needs the full
corpus and hours of GPU; on the shipped one-speaker example there is nothing to
learn and the loss curve means nothing.


In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1 if cfg.smoke else cfg.batch_size,
    per_device_eval_batch_size=1 if cfg.smoke else cfg.batch_size,
    gradient_accumulation_steps=1 if cfg.smoke else cfg.grad_accum,
    learning_rate=cfg.lr,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=not cfg.smoke,
    report_to=[],
    **({"max_steps": 2, "eval_strategy": "no", "save_strategy": "no",
        "logging_steps": 1}
       if cfg.smoke else
       {"num_train_epochs": cfg.epochs, "eval_strategy": "epoch",
        "save_strategy": "epoch", "logging_steps": 25,
        "load_best_model_at_end": True,
        "metric_for_best_model": cfg.metric_for_best, "greater_is_better": False,
        "save_total_limit": 2}),
)

trainer = Trainer(model=model, args=args, data_collator=collator,
                  train_dataset=ds["train"],
                  eval_dataset=None if cfg.smoke else ds["valid"],
                  compute_metrics=compute_metrics)
print("trainer ready ·", "SMOKE (2 steps)" if cfg.smoke else f"{cfg.epochs} epochs")


In [ ]:
trainer.train()


In [ ]:
if not cfg.smoke:
    trainer.save_model(OUT_DIR)
    processor.save_pretrained(OUT_DIR)
    print("saved ->", OUT_DIR)
    print(trainer.evaluate(ds["test"]))
else:
    print("smoke run — nothing saved, nothing evaluated")


---

## Notes

**Frame rate.** With `add_adapter=True` the encoder emits one frame per **40 ms**
instead of 20 ms. That is invisible in WER/CER and very visible in notebook 9:
forced alignment inherits the resolution, so every boundary this model places is
quantised twice as coarsely. Set `add_adapter=False` if you intend to align with
it — and expect to retune, because the CTC recipe assumes the adapter.

**Which target should you use?** Phonemes (stage 5) if you want a
transcription that survives the lack of a spelling standard, or anything
downstream that needs a phone inventory — TTS, alignment, dialectometry.
Orthography (this notebook) if you want text a Kölsch reader can read. They are
different products, not competing runs, and the honest comparison is CER on the
same speaker-disjoint split.

**What is not here.** No language-model rescoring. The source notebook has an
optional KenLM/pyctcdecode path; it is left out because it needs a Kölsch text
corpus that this repository does not ship, and a 5-gram LM trained on the same
186-word example would only memorise it.
